In [15]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import pandas as pd
import yfinance as yf
import talib

In [4]:
cl = yf.Ticker("CL=F")
data = cl.history("max")

In [6]:
iss = data.iloc[:-730]
oss = data.iloc[-730:]

In [9]:
oss.shape

(730, 7)

In [58]:
class MA_BB(Strategy):
    ema_window = 200

    def init(self):
        self.ema = self.I(talib.EMA, self.data.Close, self.ema_window)
        self.upper_band, self.middle_band, self.lower_band = self.I(
            talib.BBANDS,
            self.data.Close,
            20, 2, 2
        )

    def next(self):
        price = self.data.Close[-1]
        if not self.position:
            if price > self.ema and crossover(price, self.lower_band):
                self.buy(size=0.5)
        elif price > self.upper_band:
            self.position.close()

In [59]:
bt_iss = Backtest(iss, MA_BB, cash=10_000, commission=0.002) 

In [60]:
stats_iss = bt_iss.run()

Backtest.run:   0%|          | 0/5486 [00:00<?, ?bar/s]

In [61]:
stats_iss

Start                     2000-08-23 00:00...
End                       2023-04-18 00:00...
Duration                   8273 days 00:00:00
Exposure Time [%]                     25.4133
Equity Final [$]                   6494.91601
Equity Peak [$]                    12819.5549
Commissions [$]                     529.11498
Return [%]                          -35.05084
Buy & Hold Return [%]               178.44352
Return (Ann.) [%]                     -1.8945
Volatility (Ann.) [%]                53.38318
CAGR [%]                             -1.30597
Sharpe Ratio                         -0.03549
Sortino Ratio                        -0.11187
Calmar Ratio                         -0.02094
Alpha [%]                                 NaN
Beta                                      NaN
Max. Drawdown [%]                   -90.45664
Avg. Drawdown [%]                    -8.17352
Max. Drawdown Duration     6083 days 00:00:00
Avg. Drawdown Duration      442 days 00:00:00
# Trades                          

In [62]:
bt_iss.plot()

GridPlot(id='p1774', ...)